## Build an AnnData, then aggregate to neuron classes

awesome — here’s a clean, minimal, well-commented script that does exactly what you want:

	•	loads the CeNGEN (GSE136049) sparse matrix (.mtx) + annotations
	•	keeps neuronal cells only (and drops “Unannotated” by default)
	•	aggregates genes × neuron-class in two flavors:
	•	raw UMI sums (good for absolute abundance)
	•	CP10K + log1p means (good for embeddings / DR)
	•	maps WormBase gene IDs → transcript IDs (from the provided GTF), since your GTF lacks gene_name
	•	writes tidy CSVs (by default genes × neurons; flip orientation with one flag)

⸻

Step-by-step: what the code does

	1.	Read inputs

	•	mtx is genes × cells (rows=genes, columns=cells).
	•	genes.csv is a single column of WBGene IDs (no header).
	•	barcodes.csv is a single column of cell barcodes (no header).
	•	lookup.csv links each barcode → cell.type, tissue.type.
	•	gtf (WS273) lacks gene_name, so we map each gene_id to the first transcript_id seen for that gene; if none, we keep the WBGene ID.

	2.	Filter & align

	•	Keep rows/columns aligned to the .mtx.
	•	Filter to tissue.type == 'Neuron' and drop cell.type == 'Unannotated' (configurable).
	3.	Aggregate to neuron classes

	•	Build a one-hot matrix (cells × neuron_types).
	•	Compute raw counts per type: counts = X_neuron @ S → (genes × types).
	4.	Normalize for embeddings
	•	Per-cell library normalization to CP10K, then log1p.
	•	Take mean per neuron type (so each column is a transcriptomic profile).

	5.	Map IDs and de-duplicate

	•	Replace WBGene IDs by transcript IDs from the GTF (fallback to WBGene).
	•	If multiple WBGene IDs map to the same transcript ID, we:
	•	sum raw counts,
	•	mean the log-norm means.

	6.	Save

	•	GSE136049_genes_by_neurons_counts.csv
	•	GSE136049_genes_by_neurons_lognorm_means.csv
	•	(set ORIENTATION = "neurons_by_genes" if you want rows=neurons, cols=genes)

For dimensionality reduction / embeddings, I’d recommend the log-normalized means table.

⸻

In [5]:
"""
CeNGEN (GSE136049) → Genes × Neuron-class tables
- Produces:
    * raw UMI sums per neuron class
    * CP10K + log1p means per neuron class
- Maps WBGene IDs → transcript IDs using the provided GTF (WS273) which lacks gene_name.

Dependencies: scipy, numpy, pandas
Optional (only for speed of sparse ops): none beyond SciPy.

Place these files in your working directory (or edit the paths below):
  - GSE136049_gene_by_barcode_count_matrix_all_cells.mtx
  - GSE136049_all_cells_gene_annotations.csv            (no header)
  - GSE136049_all_cells_barcodes_column_names.csv       (no header)
  - GSE136049_cell_type_annotation_lookup_table.csv     (barcode,cell.type,tissue.type)
  - GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf[.gz]
"""

import os
import gzip
import re
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy import sparse

# -----------------------------
# Config — change paths if needed
# -----------------------------
MTX_PATH      = "data/GSE136049_gene_by_barcode_count_matrix_all_cells.mtx"
GENES_PATH    = "data/GSE136049_all_cells_gene_annotations.csv"
BARCODES_PATH = "data/GSE136049_all_cells_barcodes_column_names.csv"
LOOKUP_PATH   = "data/GSE136049_cell_type_annotation_lookup_table.csv"
GTF_PATH      = "data/GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf"  # or ...gtf.gz

# Keep only neurons and drop unannotated?
KEEP_ONLY_NEURONS = True
DROP_UNANNOTATED  = True

# Output orientation: "genes_by_neurons" (default) or "neurons_by_genes"
ORIENTATION = "genes_by_neurons"

# Normalization target sum for CP10K
TARGET_SUM = 1e4


# -----------------------------
# Utilities
# -----------------------------
def parse_gtf_gene_to_transcript(gtf_path):
    """
    Build a dict mapping: gene_id (WBGene...) → a representative transcript_id.
    Strategy: read the GTF, and take the FIRST transcript_id encountered for each gene_id.
    If a gene has no transcript entries, it won't appear in the map (we'll fallback to gene_id later).
    Supports plain or gzipped GTF.
    """
    # Open .gz or plain
    open_fn = gzip.open if gtf_path.endswith(".gz") else open

    gene_to_tx = {}
    attr_re = re.compile(r'(\S+)\s+"([^"]+)"')
    with open_fn(gtf_path, "rt") as fh:
        for line in fh:
            if not line or line.startswith("#"):
                continue
            # Only need transcript feature lines (they carry transcript_id)
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue
            feature = fields[2]
            if feature != "transcript":
                continue
            attrs = dict(attr_re.findall(fields[8]))
            gid = attrs.get("gene_id")
            tid = attrs.get("transcript_id")
            if gid and tid and gid not in gene_to_tx:
                gene_to_tx[gid] = tid
    return gene_to_tx

def aggregate_duplicates(df, how="sum"):
    """Aggregate duplicate index names (e.g., multiple genes mapping to same transcript)."""
    if how == "sum":
        return df.groupby(df.index).sum()
    elif how == "mean":
        return df.groupby(df.index).mean()
    else:
        raise ValueError("how must be 'sum' or 'mean'")

def ensure_orientation(df, orientation="genes_by_neurons"):
    return df if orientation == "genes_by_neurons" else df.T


# -----------------------------
# 1) Load the matrix & annotations
# -----------------------------
print("Reading Matrix Market file (genes × cells)...")
X = mmread(MTX_PATH).tocsr()  # shape: (n_genes, n_cells)

print("Reading gene and barcode lists (no headers)...")
genes    = pd.read_csv(GENES_PATH, header=None)[0].astype(str).values
barcodes = pd.read_csv(BARCODES_PATH, header=None)[0].astype(str).values

print("Reading barcode → cell.type / tissue.type lookup...")
lookup = pd.read_csv(LOOKUP_PATH)  # expects columns: barcode, cell.type, tissue.type

# Basic shape sanity checks
assert X.shape[0] == len(genes),   f"Gene count mismatch: {X.shape[0]} vs {len(genes)}"
assert X.shape[1] == len(barcodes),f"Cell count mismatch: {X.shape[1]} vs {len(barcodes)}"

# Reindex lookup to match the exact column order of the matrix (barcodes)
lookup = lookup.set_index("barcode").reindex(barcodes)

# -----------------------------
# 2) Filter to neurons / drop unannotated
# -----------------------------
keep_mask = np.ones(len(barcodes), dtype=bool)

if KEEP_ONLY_NEURONS:
    keep_mask &= (lookup["tissue.type"].fillna("") == "Neuron").values

if DROP_UNANNOTATED:
    keep_mask &= (lookup["cell.type"].fillna("") != "Unannotated").values

# Subset matrix columns (cells) and the aligned lookup
keep_idx = np.where(keep_mask)[0]
Xn = X[:, keep_idx]  # genes × kept_cells
lookup_n = lookup.iloc[keep_idx].copy()

print(f"Kept {Xn.shape[1]} cells out of {X.shape[1]} (neurons only = {KEEP_ONLY_NEURONS}, drop unannotated = {DROP_UNANNOTATED})")

# -----------------------------
# 3) Build one-hot of neuron classes and aggregate RAW COUNTS
# -----------------------------
# Factorize cell types into integer codes: 0..K-1
cell_types, ct_codes = np.unique(lookup_n["cell.type"].astype(str).values, return_inverse=True)

# Build indicator S (cells × neuron_types)
rows = np.arange(Xn.shape[1])              # cell indices
cols = ct_codes                             # neuron-type code per cell
data = np.ones_like(rows)
S = sparse.csr_matrix((data, (rows, cols)), shape=(Xn.shape[1], len(cell_types)))  # cells × types

# Aggregate counts to neuron types: (genes × cells) @ (cells × types) → (genes × types)
counts_mat = Xn @ S
counts_df = pd.DataFrame(
    data=np.asarray(counts_mat.todense()),
    index=genes,
    columns=cell_types
)

# -----------------------------
# 4) CP10K + log1p means per neuron class
# -----------------------------
# Per-cell library sizes (col sums) on genes×cells => sum over rows
lib_sizes = np.asarray(Xn.sum(axis=0)).ravel()  # shape: (kept_cells,)

# Avoid divide by zero: if a cell has 0 counts, set scale to 1.0 so it stays zero after scaling
safe_lib = lib_sizes.copy()
safe_lib[safe_lib == 0] = 1.0

# Scale each column to TARGET_SUM (CP10K) by right-multiplying a diagonal matrix
scale = (TARGET_SUM / safe_lib)
D = sparse.diags(scale)                      # (cells × cells)
Xn_norm = Xn @ D                             # genes × cells, CP10K scaled

# log1p on sparse values (in-place on data array)
Xn_norm = Xn_norm.tocoo(copy=True)
Xn_norm.data = np.log1p(Xn_norm.data)
Xn_norm = Xn_norm.tocsr()

# Mean per neuron class: (genes × cells) @ (cells × types) / (#cells in type)
cells_per_type = np.bincount(ct_codes, minlength=len(cell_types)).astype(float)
cells_per_type[cells_per_type == 0] = 1.0   # safety
means_mat = (Xn_norm @ S)                   # genes × types
means_mat = means_mat @ sparse.diags(1.0 / cells_per_type)

means_df = pd.DataFrame(
    data=np.asarray(means_mat.todense()),
    index=genes,
    columns=cell_types
)

# -----------------------------
# 5) Map WBGene → transcript_id (fallback to WBGene)
# -----------------------------
print("Building WBGene → transcript_id map from GTF...")
if not os.path.exists(GTF_PATH) and os.path.exists(GTF_PATH + ".gz"):
    GTF_PATH = GTF_PATH + ".gz"

gene_to_tx = parse_gtf_gene_to_transcript(GTF_PATH)

def apply_tx_index(df, how_for_dupes="sum_or_mean"):
    """
    Replace index (WBGene IDs) by transcript_id where available.
    If multiple genes share the same transcript_id, aggregate:
      - 'sum' for raw counts, 'mean' for lognorm means
    """
    idx = df.index.to_series()
    new_idx = idx.map(lambda x: gene_to_tx.get(x, x))  # fallback to WBGene if no transcript found
    out = df.copy()
    out.index = new_idx

    if how_for_dupes == "sum":
        out = aggregate_duplicates(out, how="sum")
    elif how_for_dupes == "mean":
        out = aggregate_duplicates(out, how="mean")
    else:  # auto mode: 'sum' for counts, 'mean' for means
        out_counts = aggregate_duplicates(out, how="sum")
        # If 'out' is the counts, we'll pass 'sum'; for means, we'll call with 'mean'
        return out_counts  # caller will use appropriate mode

    return out

# Raw counts (sum duplicates)
counts_tx = apply_tx_index(counts_df, how_for_dupes="sum")
# Lognorm means (mean duplicates)
means_tx  = apply_tx_index(means_df,  how_for_dupes="mean")

# -----------------------------
# 6) Orient & save
# -----------------------------
counts_out = ensure_orientation(counts_tx, ORIENTATION)
means_out  = ensure_orientation(means_tx,  ORIENTATION)

suffix = "genes_by_neurons" if ORIENTATION == "genes_by_neurons" else "neurons_by_genes"
counts_path = f"GSE136049_{suffix}_counts.csv"
means_path  = f"GSE136049_{suffix}_lognorm_means.csv"

counts_out.to_csv(counts_path)
means_out.to_csv(means_path)

print("Done.")
print(f"  Raw counts:      {counts_path}  (shape={counts_out.shape})")
print(f"  Lognorm means:   {means_path}   (shape={means_out.shape})")

Reading Matrix Market file (genes × cells)...
Reading gene and barcode lists (no headers)...
Reading barcode → cell.type / tissue.type lookup...
Kept 70296 cells out of 100955 (neurons only = True, drop unannotated = True)
Building WBGene → transcript_id map from GTF...
Done.
  Raw counts:      GSE136049_genes_by_neurons_counts.csv  (shape=(22469, 130))
  Lognorm means:   GSE136049_genes_by_neurons_lognorm_means.csv   (shape=(22469, 130))


Practical notes

	•	Which table to feed into embeddings?
Use the log-normalized means file. It’s far more comparable across neuron classes, and it’ll behave better with PCA/UMAP/TSNE.

	•	Orientation:
If you want rows=neurons, set ORIENTATION = "neurons_by_genes".

	•	Keeping all cells:
Set KEEP_ONLY_NEURONS = False and/or DROP_UNANNOTATED = False if you want everything.

	•	Transcript ID mapping:
We map WBGene → first transcript_id encountered. If you’d rather map to protein IDs (protein_id) or keep WBGene IDs, say the word and I’ll swap that in the parser.

	•	Duplicates after mapping:
Some genes can share a transcript label (rare but possible depending on annotation). We sum raw counts and average lognorm means in those cases.

	•	Out-of-memory concerns:
This works fully sparse and should be OK on a typical laptop. If you hit memory issues, I can give you a chunked variant.

⸻

